In [ ]:
from __future__ import annotations

import gc
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except Exception:  # pragma: no cover
    torch = None


OUTER_FOLDS = (1, 2, 3, 4, 5)
MODEL_FAMILY = "tabpfn"
MODEL_ID = "tabpfn_v26_core14"
RUN_ROLE = "default"
TARGET_COL = "Target_Log"
TABPFN_N_ESTIMATORS = 8
TABPFN_SUBSAMPLE_SAMPLES = 2000
SEED = 42
BATCH_SIZE = 512
PREDICT_PROGRESS_EVERY = 2
EXPECTED_DRIVER_FEATURE_COUNT = 10
EXPECTED_ROUTE_FEATURES = ["WasteRoute_AW", "WasteRoute_CDW", "WasteRoute_IW", "WasteRoute_MSW"]
EXPECTED_CORE_FEATURE_COUNT = EXPECTED_DRIVER_FEATURE_COUNT + len(EXPECTED_ROUTE_FEATURES)


build_prediction_frame = None
compute_regression_metrics = None
summarize_fold_metrics = None
save_model_artifact = None
TabPFNRegressor = None
ModelVersion = None

In [ ]:
def resolve_code_dir() -> Path:
    if "__file__" in globals():
        return Path(__file__).resolve().parent
    code_dir = Path.cwd().resolve()
    if code_dir.name != "1_Code":
        raise RuntimeError("current working directory must be Step6_TabPFNExplainability/1_Code")
    return code_dir


CODE_DIR = resolve_code_dir()
STEP_ROOT = CODE_DIR.parent
RELEASE_ROOT = STEP_ROOT.parent
STEP4_ROOT = RELEASE_ROOT / "Step4_ModelInputs" / "2_Results"
STEP5_ROOT = RELEASE_ROOT / "Step5_ModelComparison"
RESULT_DIR = STEP_ROOT / "2_Results"
ARTIFACT_DIR = RESULT_DIR / "1_core_feature_ablation_artifacts"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_PATH = RESULT_DIR / "1_core_feature_ablation_predictions_outer_test.csv"
METRICS_PATH = RESULT_DIR / "1_core_feature_ablation_metrics_by_outer_fold.csv"
SUMMARY_PATH = RESULT_DIR / "1_core_feature_ablation_metrics_mean_std.csv"
COMPARISON_PATH = RESULT_DIR / "1_core_vs_full_tabpfn_summary.csv"
FEATURE_CONTRACT_PATH = RESULT_DIR / "1_core_feature_ablation_feature_contract.csv"
ARTIFACT_MANIFEST_PATH = RESULT_DIR / "1_core_feature_ablation_model_artifact_manifest.csv"

if not STEP5_ROOT.is_dir():
    raise FileNotFoundError(f"cannot resolve Step5 root from {STEP5_ROOT}")
if not STEP4_ROOT.is_dir():
    raise FileNotFoundError(f"cannot resolve Step4 results from {STEP4_ROOT}")

print({"step": "paths_ready", "result_dir": str(RESULT_DIR), "artifact_dir": str(ARTIFACT_DIR)})

In [ ]:
def initialize_runtime() -> None:
    global build_prediction_frame
    global compute_regression_metrics
    global summarize_fold_metrics
    global save_model_artifact
    global TabPFNRegressor
    global ModelVersion

    step5_code_dir = STEP5_ROOT / "1_Code"
    if str(step5_code_dir) not in sys.path:
        sys.path.insert(0, str(step5_code_dir))

    from _step5_shared import (
        TARGET_COL as shared_target_col,
        build_prediction_frame as shared_build_prediction_frame,
        compute_regression_metrics as shared_compute_regression_metrics,
        save_model_artifact as shared_save_model_artifact,
        summarize_fold_metrics as shared_summarize_fold_metrics,
    )

    if shared_target_col != TARGET_COL:
        raise RuntimeError(f"unexpected target column from Step5 shared module: {shared_target_col}")

    from tabpfn import TabPFNRegressor as runtime_tabpfn_regressor
    from tabpfn.constants import ModelVersion as runtime_model_version

    supported_versions = [name for name in dir(runtime_model_version) if name.startswith("V")]
    if "V2_6" not in supported_versions:
        raise RuntimeError("tabpfn runtime must expose ModelVersion.V2_6; version fallback is not allowed")

    if torch is not None:
        torch.set_num_threads(max(1, min(8, torch.get_num_threads())))

    build_prediction_frame = shared_build_prediction_frame
    compute_regression_metrics = shared_compute_regression_metrics
    save_model_artifact = shared_save_model_artifact
    summarize_fold_metrics = shared_summarize_fold_metrics
    TabPFNRegressor = runtime_tabpfn_regressor
    ModelVersion = runtime_model_version
    print({"step": "runtime_ready", "tabpfn_versions": supported_versions})


initialize_runtime()

In [ ]:
def require_file(path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError(str(path))
    return path


def read_feature_list(path: Path) -> list[str]:
    frame = pd.read_csv(require_file(path))
    if "feature" not in frame.columns:
        raise KeyError(f"feature column missing from {path}")
    features = frame["feature"].astype(str).tolist()
    if len(features) != len(set(features)):
        raise RuntimeError(f"feature contract contains duplicates: {path}")
    return features


def expected_driver_features_from_step4() -> list[str]:
    driver_features = read_feature_list(STEP4_ROOT / "0_feature_contract_10feature.csv")
    if len(driver_features) != EXPECTED_DRIVER_FEATURE_COUNT:
        raise RuntimeError(f"Step4 locked ten-feature contract must contain 10 driver features, got {len(driver_features)}")
    return driver_features


def expected_core_features_from_step4() -> list[str]:
    fold_feature_lists = []
    for outer_fold in OUTER_FOLDS:
        path = STEP4_ROOT / f"fold_{int(outer_fold)}" / "ten_feature_unified" / "processed" / "0_feature_columns.csv"
        fold_feature_lists.append(read_feature_list(path))
    first = fold_feature_lists[0]
    drift = {int(fold): features for fold, features in zip(OUTER_FOLDS, fold_feature_lists) if features != first}
    if drift:
        raise RuntimeError(f"Step4 ten-feature unified feature contract differs across folds: {drift}")
    return first


EXPECTED_DRIVER_FEATURES = expected_driver_features_from_step4()
EXPECTED_CORE_FEATURES = expected_core_features_from_step4()
if len(EXPECTED_CORE_FEATURES) != EXPECTED_CORE_FEATURE_COUNT:
    raise RuntimeError(f"core ablation contract must contain 14 features, got {len(EXPECTED_CORE_FEATURES)}")
if EXPECTED_CORE_FEATURES[:EXPECTED_DRIVER_FEATURE_COUNT] != EXPECTED_DRIVER_FEATURES:
    raise RuntimeError(
        "Step4 ten-feature unified driver contract does not match Step4 locked 10-feature contract: "
        f"unified_drivers={EXPECTED_CORE_FEATURES[:EXPECTED_DRIVER_FEATURE_COUNT]}, locked_drivers={EXPECTED_DRIVER_FEATURES}"
    )
if EXPECTED_CORE_FEATURES[EXPECTED_DRIVER_FEATURE_COUNT:] != EXPECTED_ROUTE_FEATURES:
    raise RuntimeError(
        "Step4 ten-feature unified route features do not match expected route features: "
        f"actual={EXPECTED_CORE_FEATURES[EXPECTED_DRIVER_FEATURE_COUNT:]}, expected={EXPECTED_ROUTE_FEATURES}"
    )

pd.DataFrame({"feature": EXPECTED_CORE_FEATURES}).to_csv(FEATURE_CONTRACT_PATH, index=False)
print({"step": "core_feature_contract_ready", "feature_count": len(EXPECTED_CORE_FEATURES)})
print(pd.DataFrame({"feature": EXPECTED_CORE_FEATURES}).to_string(index=False))

In [ ]:
def load_core_feature_processed(outer_fold: int) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    processed_dir = STEP4_ROOT / f"fold_{int(outer_fold)}" / "ten_feature_unified" / "processed"
    train_df = pd.read_csv(require_file(processed_dir / "0_train_processed.csv"))
    test_df = pd.read_csv(require_file(processed_dir / "0_test_processed.csv"))
    feature_cols = read_feature_list(processed_dir / "0_feature_columns.csv")

    if feature_cols != EXPECTED_CORE_FEATURES:
        raise RuntimeError(
            f"fold {outer_fold} ten-feature unified contract drift: "
            f"expected={EXPECTED_CORE_FEATURES}, actual={feature_cols}"
        )

    required_columns = ["row_uid", "Country Code", "Year", "WasteFlag", "outer_fold", "Waste_Generation_t", TARGET_COL, *feature_cols]
    missing_train = [column for column in required_columns if column not in train_df.columns]
    missing_test = [column for column in required_columns if column not in test_df.columns]
    if missing_train or missing_test:
        raise KeyError(f"fold {outer_fold} input missing columns: train={missing_train}, test={missing_test}")
    test_outer_folds = sorted(test_df["outer_fold"].dropna().astype(int).unique().tolist())
    if test_outer_folds != [int(outer_fold)]:
        raise RuntimeError(f"fold {outer_fold} test_df has unexpected outer_fold values: {test_outer_folds}")
    if train_df["row_uid"].astype(str).duplicated().any():
        raise RuntimeError(f"fold {outer_fold} train_df contains duplicate row_uid values")
    if test_df["row_uid"].astype(str).duplicated().any():
        raise RuntimeError(f"fold {outer_fold} test_df contains duplicate row_uid values")
    overlap = set(train_df["row_uid"].astype(str)).intersection(set(test_df["row_uid"].astype(str)))
    if overlap:
        raise RuntimeError(f"fold {outer_fold} train/test row_uid overlap detected: overlap_count={len(overlap)}")
    return train_df, test_df, feature_cols


def prepare_tabpfn_feature_frames(
    train_df: pd.DataFrame,
    other_df: pd.DataFrame,
    feature_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    x_train = train_df.loc[:, list(feature_cols)].apply(pd.to_numeric, errors="coerce")
    x_other = other_df.loc[:, list(feature_cols)].apply(pd.to_numeric, errors="coerce")
    medians = x_train.median(axis=0).fillna(0.0)
    x_train = x_train.fillna(medians)
    x_other = x_other.fillna(medians)
    y_train = pd.Series(pd.to_numeric(train_df[TARGET_COL], errors="raise"), index=train_df.index).astype(float)
    if not np.isfinite(x_train.to_numpy(dtype=float)).all():
        raise ValueError("training feature matrix contains non-finite values")
    if not np.isfinite(x_other.to_numpy(dtype=float)).all():
        raise ValueError("evaluation feature matrix contains non-finite values")
    if not np.isfinite(y_train.to_numpy(dtype=float)).all():
        raise ValueError("training target contains non-finite values")
    return x_train, x_other, y_train


print({"step": "input_loader_ready", "folds": list(OUTER_FOLDS)})

In [ ]:
def build_tabpfn_regressor(random_seed: int):
    device = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
    return TabPFNRegressor.create_default_for_version(
        ModelVersion.V2_6,
        n_estimators=TABPFN_N_ESTIMATORS,
        device=device,
        fit_mode="low_memory",
        memory_saving_mode="auto",
        ignore_pretraining_limits=True,
        inference_config={"SUBSAMPLE_SAMPLES": TABPFN_SUBSAMPLE_SAMPLES},
        random_state=int(random_seed),
    )


def predict_in_batches(model, features: pd.DataFrame, batch_size: int = BATCH_SIZE, progress_every: int = PREDICT_PROGRESS_EVERY) -> np.ndarray:
    n_rows = len(features)
    if n_rows == 0:
        return np.asarray([], dtype=float)
    if batch_size is None or int(batch_size) <= 0 or int(batch_size) >= n_rows:
        return np.asarray(model.predict(features), dtype=float).reshape(-1)
    predictions: list[np.ndarray] = []
    n_batches = (n_rows - 1) // int(batch_size) + 1
    for batch_idx in range(n_batches):
        start = batch_idx * int(batch_size)
        end = min((batch_idx + 1) * int(batch_size), n_rows)
        batch_features = features.iloc[start:end]
        batch_pred = np.asarray(model.predict(batch_features), dtype=float).reshape(-1)
        predictions.append(batch_pred)
        if progress_every and (((batch_idx + 1) % int(progress_every) == 0) or (batch_idx + 1 == n_batches)):
            print(f"Prediction batch {batch_idx + 1}/{n_batches} ({end}/{n_rows} rows)", flush=True)
    return np.concatenate(predictions, axis=0)


def clear_runtime_memory() -> None:
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


print({"step": "model_helpers_ready", "n_estimators": TABPFN_N_ESTIMATORS, "subsample_samples": TABPFN_SUBSAMPLE_SAMPLES})

In [ ]:
def run_one_fold_ablation(outer_fold: int) -> dict[str, object]:
    train_df, test_df, feature_cols = load_core_feature_processed(outer_fold)
    x_train, x_test, y_train = prepare_tabpfn_feature_frames(train_df, test_df, feature_cols)
    print(
        {
            "step": "fold_start",
            "outer_fold": int(outer_fold),
            "train_rows": int(len(x_train)),
            "test_rows": int(len(x_test)),
            "features": int(len(feature_cols)),
        },
        flush=True,
    )
    model = build_tabpfn_regressor(random_seed=SEED)
    model.fit(x_train, y_train.to_numpy(dtype=float))
    prediction_log = predict_in_batches(model, x_test, batch_size=BATCH_SIZE, progress_every=PREDICT_PROGRESS_EVERY)
    prediction_frame = build_prediction_frame(
        test_df=test_df,
        prediction_log=prediction_log,
        outer_fold=int(outer_fold),
        model_family=MODEL_FAMILY,
        model_id=MODEL_ID,
        run_role=RUN_ROLE,
    )
    metric_row = {
        "outer_fold": int(outer_fold),
        "model_family": MODEL_FAMILY,
        "model_id": MODEL_ID,
        "run_role": RUN_ROLE,
        **compute_regression_metrics(prediction_frame),
    }
    artifact_path = ARTIFACT_DIR / f"fold_{int(outer_fold)}" / f"{MODEL_ID}_{RUN_ROLE}.joblib"
    save_model_artifact(model, artifact_path)
    if not artifact_path.is_file():
        raise RuntimeError(f"artifact path does not exist after save: {artifact_path}")
    artifact_row = {
        "outer_fold": int(outer_fold),
        "model_family": MODEL_FAMILY,
        "model_id": MODEL_ID,
        "run_role": RUN_ROLE,
        "feature_count": len(feature_cols),
        "seed": SEED,
        "tabpfn_n_estimators": TABPFN_N_ESTIMATORS,
        "tabpfn_subsample_samples": TABPFN_SUBSAMPLE_SAMPLES,
        "batch_size": BATCH_SIZE,
        "artifact_path": str(artifact_path),
    }
    del model
    clear_runtime_memory()
    print({"step": "fold_completed", "outer_fold": int(outer_fold), **metric_row}, flush=True)
    return {"prediction_frame": prediction_frame, "metric_row": metric_row, "artifact_row": artifact_row}


fold_results_by_fold: dict[int, dict[str, object]] = {}
print({"step": "fold_runner_ready"})

In [ ]:
fold_results_by_fold[1] = run_one_fold_ablation(1)

In [ ]:
fold_results_by_fold[2] = run_one_fold_ablation(2)

In [ ]:
fold_results_by_fold[3] = run_one_fold_ablation(3)

In [ ]:
fold_results_by_fold[4] = run_one_fold_ablation(4)

In [ ]:
fold_results_by_fold[5] = run_one_fold_ablation(5)

In [ ]:
def write_core_ablation_outputs() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    completed_folds = sorted(int(fold) for fold in fold_results_by_fold)
    missing_folds = [int(fold) for fold in OUTER_FOLDS if int(fold) not in fold_results_by_fold]
    if missing_folds:
        raise RuntimeError(f"cannot finalize core ablation before all folds are completed: missing={missing_folds}")
    prediction_frames = [fold_results_by_fold[fold]["prediction_frame"] for fold in completed_folds]
    metric_rows = [fold_results_by_fold[fold]["metric_row"] for fold in completed_folds]
    artifact_rows = [fold_results_by_fold[fold]["artifact_row"] for fold in completed_folds]
    predictions = pd.concat(prediction_frames, ignore_index=True)
    metrics = pd.DataFrame(metric_rows)
    summary = summarize_fold_metrics(metrics)
    artifacts = pd.DataFrame(artifact_rows)
    predictions.to_csv(PREDICTION_PATH, index=False)
    metrics.to_csv(METRICS_PATH, index=False)
    summary.to_csv(SUMMARY_PATH, index=False)
    artifacts.to_csv(ARTIFACT_MANIFEST_PATH, index=False)
    print({"step": "core_ablation_outputs_written", "prediction_rows": int(len(predictions)), "metric_rows": int(len(metrics))})
    print(summary.to_string(index=False))
    return predictions, metrics, summary


core_predictions, core_metrics, core_summary = write_core_ablation_outputs()

In [ ]:
def load_full_tabpfn_summary() -> pd.DataFrame:
    full_path = STEP5_ROOT / "2_Results" / "3_tabpfn_family_compare" / "3_tabpfn_metrics_by_model_mean_std.csv"
    full = pd.read_csv(require_file(full_path))
    full = full[(full["model_id"].astype(str) == "tabpfn_v26") & (full["run_role"].astype(str) == "default")].copy()
    if len(full) != 1:
        raise RuntimeError("expected exactly one Step5 full-feature tabpfn_v26 default summary row")
    full["feature_set"] = "full_feature_33"
    full["feature_count"] = 33
    return full


def build_core_vs_full_summary(core_summary_frame: pd.DataFrame) -> pd.DataFrame:
    full = load_full_tabpfn_summary()
    core = core_summary_frame.copy()
    core["feature_set"] = "core_feature_14"
    core["feature_count"] = EXPECTED_CORE_FEATURE_COUNT
    comparison = pd.concat([full, core], ignore_index=True, sort=False)
    ordered_columns = [
        "feature_set",
        "feature_count",
        "model_family",
        "model_id",
        "run_role",
        "WAPE_micro_mean",
        "WAPE_micro_std",
        "R2_log_mean",
        "R2_log_std",
        "WAPE_macro_mean",
        "WAPE_macro_std",
        "R2_original_mean",
        "R2_original_std",
        "MAE_original_mean",
        "MAE_original_std",
    ]
    missing_columns = [column for column in ordered_columns if column not in comparison.columns]
    if missing_columns:
        raise RuntimeError(f"comparison summary missing expected columns: {missing_columns}")
    comparison = comparison.loc[:, ordered_columns]
    full_wape = float(comparison.loc[comparison["feature_set"] == "full_feature_33", "WAPE_micro_mean"].iloc[0])
    core_wape = float(comparison.loc[comparison["feature_set"] == "core_feature_14", "WAPE_micro_mean"].iloc[0])
    full_r2 = float(comparison.loc[comparison["feature_set"] == "full_feature_33", "R2_log_mean"].iloc[0])
    core_r2 = float(comparison.loc[comparison["feature_set"] == "core_feature_14", "R2_log_mean"].iloc[0])
    if full_wape <= 0:
        raise RuntimeError(f"full-feature WAPE_micro_mean must be positive for ratio comparison, got {full_wape}")
    comparison["delta_vs_full_WAPE_micro_mean"] = comparison["WAPE_micro_mean"] - full_wape
    comparison["delta_vs_full_R2_log_mean"] = comparison["R2_log_mean"] - full_r2
    comparison["relative_WAPE_micro_vs_full"] = comparison["WAPE_micro_mean"] / full_wape
    comparison.to_csv(COMPARISON_PATH, index=False)
    print({"step": "comparison_written", "core_wape_delta": core_wape - full_wape, "core_r2_delta": core_r2 - full_r2})
    print(comparison.to_string(index=False))
    return comparison


core_vs_full_summary = build_core_vs_full_summary(core_summary)

In [ ]:
def assert_expected_outputs() -> None:
    expected_paths = [
        PREDICTION_PATH,
        METRICS_PATH,
        SUMMARY_PATH,
        COMPARISON_PATH,
        FEATURE_CONTRACT_PATH,
        ARTIFACT_MANIFEST_PATH,
    ]
    missing = [str(path) for path in expected_paths if not path.is_file()]
    if missing:
        raise RuntimeError(f"missing expected core ablation output files: {missing}")
    metrics = pd.read_csv(METRICS_PATH)
    if set(metrics["outer_fold"].astype(int)) != set(OUTER_FOLDS):
        raise RuntimeError("core ablation metrics must cover folds 1..5")
    artifacts = pd.read_csv(ARTIFACT_MANIFEST_PATH)
    if set(artifacts["outer_fold"].astype(int)) != set(OUTER_FOLDS):
        raise RuntimeError("core ablation artifact manifest must cover folds 1..5")
    for artifact_path in artifacts["artifact_path"].astype(str):
        require_file(Path(artifact_path))
    signature_cols = ["feature_count", "seed", "tabpfn_n_estimators", "tabpfn_subsample_samples", "batch_size"]
    signature_counts = artifacts[signature_cols].drop_duplicates().shape[0]
    if signature_counts != 1:
        raise RuntimeError("core ablation artifacts must share one runtime signature across folds")
    predictions = pd.read_csv(PREDICTION_PATH, usecols=["outer_fold", "row_uid", "Prediction_Log", "Prediction_Raw"])
    if predictions.empty:
        raise RuntimeError("core ablation predictions must not be empty")
    if set(predictions["outer_fold"].astype(int)) != set(OUTER_FOLDS):
        raise RuntimeError("core ablation predictions must cover folds 1..5")
    if predictions["row_uid"].astype(str).duplicated().any():
        raise RuntimeError("core ablation predictions contain duplicate row_uid values")
    expected_row_uids = set()
    expected_counts = {}
    for outer_fold in OUTER_FOLDS:
        test_path = STEP4_ROOT / f"fold_{int(outer_fold)}" / "ten_feature_unified" / "processed" / "0_test_processed.csv"
        test_rows = pd.read_csv(require_file(test_path), usecols=["row_uid"])
        expected_counts[int(outer_fold)] = int(len(test_rows))
        expected_row_uids.update(test_rows["row_uid"].astype(str))
    actual_counts = predictions.groupby("outer_fold").size().astype(int).to_dict()
    if actual_counts != expected_counts:
        raise RuntimeError(f"prediction row counts by fold mismatch: actual={actual_counts}, expected={expected_counts}")
    actual_row_uids = set(predictions["row_uid"].astype(str))
    if actual_row_uids != expected_row_uids:
        raise RuntimeError("core ablation prediction row_uid set does not match Step4 ten-feature test rows")
    full_prediction_path = STEP5_ROOT / "2_Results" / "3_tabpfn_family_compare" / "3_tabpfn_predictions_outer_test.csv"
    full_row_uids = set(pd.read_csv(require_file(full_prediction_path), usecols=["row_uid"])["row_uid"].astype(str))
    if actual_row_uids != full_row_uids:
        raise RuntimeError("core ablation prediction row_uid set does not match Step5 full-feature TabPFN predictions")
    if not np.isfinite(predictions[["Prediction_Log", "Prediction_Raw"]].to_numpy(dtype=float)).all():
        raise RuntimeError("core ablation predictions contain non-finite values")
    print({"step": "output_assertions_passed", "prediction_rows": int(len(predictions)), "folds": sorted(metrics["outer_fold"].astype(int).unique())})


assert_expected_outputs()